In [ ]:
# IMPORTS

# 1. Symbolic Mathematics (Analytical Derivations & Units)
import sympy as sp
from IPython.display import display, Math # For rendering LaTeX beautifully in Jupyter

# 2. Numerical Arrays and Matrices
import numpy as np

# 3. Scientific Computing (Integration and Eigenvalue Solvers)
from scipy.integrate import quad
from scipy.linalg import eigh

# 4. Visualization (Convergence and Error Graphs)
import matplotlib.pyplot as plt

In [ ]:
# CONFIGURATION, CONSTANTS & FLAGS

# --- Physical Parameters ---
# Since we are using Natural Units (hbar = m = omega = 1), 
# we don't define them here to avoid floating point errors.
L = 10.0  # The domain size of the infinite well [-L/2, L/2]

# --- Variational Parameters ---
N_BASIS = 50  # The size of the basis (k = 1, 2, ..., N_BASIS)
N_STATES = 5  # How many low-lying energy levels to track (e.g., ground, 1st, 2nd) Important! This should be <= N_BASIS, preferably N_STATES << N_BASIS for better accuracy.

# --- Execution Flags ---
# These control the logic flow, making debugging much easier.
DEBUG_PRINT = True       # Set to True to print matrix values (keep N_BASIS small if True!)
PLOT_ERROR = True        # Flag to trigger the matplotlib convergence graph
PLOT_WAVEFUNCTION = True # Flag to trigger visualizing the final wavefunctions

# --- Numerical Methods ---
integration_method = 'analytical'  # Options: 'analytical', 'scipy', 'custom' (custom is the code I wrote) # TODO: change custom to a detailed description of which method (e.g. simpsons or trapezoidal) and why it is used. Also, add a note that this is not recommended for large N_BASIS due to numerical instability.
eigenvalue_method = 'scipy'        # Options: 'scipy', 'numpy', 'custom' (custom is the code I wrote) # TODO: change custom to a detailed description of which method (e.g. QR algorithm or Jacobi iteration) and why it is used. Also, add a note that this is not recommended for large N_BASIS due to numerical instability.

In [ ]:
# The basis functions of the infinite square well (of size L from [-L/2, L/2])
def basis_function(n, x, L):
    """
    Evaluates the n-th infinite square well basis function at position x.
    """
    normalization = np.sqrt(2.0 / L)
    argument = (n * np.pi * (x + L / 2.0)) / L
    
    return normalization * np.sin(argument)

In [ ]:
# NUMERICAL INTEGRATION METHODS

def integral_solve_using_trapezoidal_rule(f, a, b, N, *args):
    """
    Solves the definite integral of a function using the trapezoidal rule.

    Parameters
    ----------
    f : callable
        The objective function to be integrated. Must accept the integration variable
        as its first argument, followed by any additional arguments (*args).
    a : float
        The lower limit of integration.
    b : float
        The upper limit of integration.
    N : int
        The number of sub-intervals to divide the integration range into.
    *args : tuple, optional
        Additional arguments to pass to the objective function `f`.

    Returns
    -------
    float
        The estimated numerical value of the integral. The error scales as O(h^2).
    """
    h = (b - a) / N
    s = f(a, *args) / 2.0
    
    for i in range(1, N):
        s += f(a + h * i, *args)
        
    s += f(b, *args) / 2.0
    s *= h
    
    return s


def integral_solve_using_romberg_method(f, a, b, N, *args):
    """
    Solves the definite integral of a function using Romberg's method.
    
    This method extrapolates a more accurate estimate from two trapezoidal 
    approximations, significantly reducing the approximation error.

    Parameters
    ----------
    f : callable
        The objective function to be integrated. Must accept the integration variable
        as its first argument, followed by any additional arguments (*args).
    a : float
        The lower limit of integration.
    b : float
        The upper limit of integration.
    N : int
        The initial number of sub-intervals. The method will also evaluate 
        the function at 2*N intervals internally.
    *args : tuple, optional
        Additional arguments to pass to the objective function `f`.

    Returns
    -------
    float
        The estimated numerical value of the integral. The error scales as O(h^4).
    """
    # Calculate S_n using the trapezoidal rule
    s_n = integral_solve_using_trapezoidal_rule(f, a, b, N, *args)

    # Calculate S_2n using the trapezoidal rule with double the intervals
    s_2n = integral_solve_using_trapezoidal_rule(f, a, b, 2 * N, *args)

    # Romberg extrapolation formula
    solution = (4.0 / 3.0) * s_2n - (1.0 / 3.0) * s_n
    
    return solution

def integral_solve_using_simpsons_rule(f, a, b, N, *args):
    """
    Solves the definite integral of a function using Simpson's 1/3 rule.

    Parameters
    ----------
    f : callable
        The objective function to be integrated. 
    a : float
        The lower limit of integration.
    b : float
        The upper limit of integration.
    N : int
        The number of sub-intervals. If an odd integer is provided, 
        it will be automatically incremented to the next even integer.
    *args : tuple, optional
        Additional arguments to pass to the objective function `f`.

    Returns
    -------
    float
        The estimated numerical value of the integral. The error scales as O(h^4).
    """
    # Force N to be an integer first, just in case a float was passed
    N = int(N)
    
    # Automatically adjust odd numbers to the nearest even number (rounding up)
    if N % 2 != 0:
        N += 1
        print(f"Warning: Simpson's rule requires an even N. Automatically adjusted N to {N}.")

    h = (b - a) / N
    s = f(a, *args) + f(b, *args)

    # Sum the odd-indexed terms (multiplied by 4)
    for i in range(1, N, 2):
        s += 4.0 * f(a + i * h, *args)
        
    # Sum the even-indexed terms (multiplied by 2)
    for i in range(2, N - 1, 2):
        s += 2.0 * f(a + i * h, *args)

    return (h / 3.0) * s

In [ ]:
# Calculate the Hamiltonian matrix elements (solve many integrals)

H = np.zeros((N_BASIS, N_BASIS))

# Hamiltonian is hermitian, so H_i,j = H_j,i*. We only need to calculate N_BASIS + (N_BASIS - 1) + (N_BASIS - 2) + ... + 1 = N_BASIS * (N_BASIS + 1) / 2 unique elements. The rest can be filled in by symmetry.

# H = T + V, where T is the kinetic energy term and V is the potential energy term.
for i in range(N_BASIS):
    for j in range(i, N_BASIS):

        # shift indices to match basis functions (important!)
        n = i + 1
        m = j + 1

        # the kinetic energy term is known from analytical calculations (see above calculations #TODO: add calculations)
        T = (((np.pi * n) / L)**2)/ 2 if n == m else 0.0

        # the potential energy term can be calculated analytically or numerically
        if integration_method == 'analytical':
            # Use the analytical formula derived using SymPy
            V = calculate_V_analytical(n, m, L)
        elif integration_method == 'simpsons':
            # Use the custom numerical integration method I implemented using Simpson's rule
            V = custom_simpson_integral(integrand_V, -L/2, L/2, n, m, L) # TODO: change this signature to the one from teh other project
        elif integration_method == 'trapezoidal':
            # Use the custom numerical integration method I implemented using the trapezoidal rule
            V = custom_trapezoidal_integral(integrand_V, -L/2, L/2, n, m, L) # TODO: change this signature to the one from teh other project
        elif integration_method == 'scipy':
            V, _ = quad(integrand_V, -L/2, L/2, args=(n, m, L), limit=100)
        else:
            raise ValueError("Invalid integration method selected!")
        
        H[i, j] = T + V
        H[j, i] = H[i, j]  # Fill in the symmetric element (base functions are real, so H is symmetric)

In [ ]:
# Maybe calculate S matrix here for generality, but for the infinite square well, the basis functions are orthonormal, so S = I (identity matrix).

In [ ]:
# Solve secular equation, which is just the eigenvalue problem for the Hamiltonian matrix H.

if eigenvalue_method == 'scipy':
    # Use SciPy's eigh function to solve the eigenvalue problem
    energies, wavefunctions = eigh(H)

elif eigenvalue_method == 'QR':
    # Use a custom QR algorithm to solve the eigenvalue problem
    energies, wavefunctions = custom_qr_algorithm(H)

elif eigenvalue_method == 'Jacobi':
    # Use a custom Jacobi iteration to solve the eigenvalue problem
    energies, wavefunctions = custom_jacobi_iteration(H)

else:
    raise ValueError("Invalid eigenvalue method selected!")